# tbot in AWS Lambda → X509-SVID → passwordless Postgres

A Go Lambda obtains a SPIFFE **X509-SVID** from Teleport Workload Identity and uses it as its
TLS **client certificate** to log in to PostgreSQL. The function stores **no secrets**: no DB
password, no Teleport join token secret, no long-lived certificates.

```
INIT    tbot extension ──IAM join (signed sts:GetCallerIdentity)──▶ Teleport
                       ◀──────────────── bot identity (short-lived)
                       serves SPIFFE Workload API on unix:///tmp/tbot/workload.sock

INVOKE  handler ──FetchX509SVID──▶ tbot ──Unix attestation (pid/uid/binary_path)──▶ Teleport
                                        ◀── X509-SVID  CN=lambda_svid_demo, TTL 1h
        handler ──TLS + client cert──▶ Postgres
                                        chain to Teleport SPIFFE CA ✔  CN == role ✔
                                        current_user = lambda_svid_demo
```

**What this notebook proves**

| Claim | Where you see it |
|---|---|
| Identity comes from AWS, not a secret | Teleport's join event records the Lambda's assumed-role ARN |
| The SVID is short-lived and per-invoke | `not_after` ≤ 1 h, and a different `serial` on every invoke |
| Postgres authenticates the certificate | `ssl_client_dn` is populated, `current_user = lambda_svid_demo` |
| Nothing secret is in the function | The env holds only names — proxy address, token name, WI name |

**Run order matters.** Every cell is idempotent: re-running one converges rather than
duplicating. Run them top to bottom; re-run any of them safely.

> **Not RDS.** RDS and Aurora Postgres do not support client-certificate authentication.
> This demo uses self-managed Postgres on EC2. Other databases with X.509 auth include
> CockroachDB, MongoDB, and MySQL `REQUIRE SUBJECT`.

## 0. Notebook setup

Loads the helpers every later cell depends on. Re-run this after restarting the kernel.

In [ ]:
import datetime, json as _json, os, re, shutil, socket, subprocess, sys, time
from IPython.display import display, Markdown

REPO_DIR = os.path.dirname(os.path.abspath("demo-walkthrough.ipynb"))
DB_DIR   = os.path.join(REPO_DIR, "terraform", "db")
TF_DIR   = os.path.join(REPO_DIR, "terraform")


class _RunResult:
    """Wraps CompletedProcess so Jupyter does not echo an ugly repr."""

    def __init__(self, proc):
        object.__setattr__(self, "_proc", proc)

    def __getattr__(self, name):
        return getattr(object.__getattribute__(self, "_proc"), name)

    def _ipython_display_(self, **kwargs):
        pass


def run(cmd, *, cwd=REPO_DIR, check=True, capture=True, delay=0, quiet=False):
    """Run a shell command, echo it, print its output."""
    if not quiet:
        print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=capture,
                       text=True, env=os.environ)
    if capture:
        for stream in (r.stdout, r.stderr):
            if stream:
                print(stream, end="" if stream.endswith("\n") else "\n")
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed (exit {r.returncode}): {cmd}")
    if delay:
        time.sleep(delay)
    return _RunResult(r)


def out(cmd, *, cwd=REPO_DIR, check=True):
    """Run silently, return stripped stdout."""
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True,
                       text=True, env=os.environ)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd}\n{r.stderr.strip()}")
    return r.stdout.strip()


def ssh(remote_cmd, *, check=True, show=True):
    """Run a command on the Postgres host. Uses a list, so no shell quoting games."""
    argv = SSH_ARGV + [remote_cmd]
    if show:
        print(f"$ ssh {DB_HOST} {remote_cmd}")
    r = subprocess.run(argv, capture_output=True, text=True, env=os.environ)
    if r.stdout:
        print(r.stdout, end="" if r.stdout.endswith("\n") else "\n")
    if check and r.returncode != 0:
        raise RuntimeError(f"ssh failed (exit {r.returncode}): {r.stderr.strip()}")
    return r


def load_dotenv(path, *, into_environ=False):
    """Minimal .env parser, no external dependencies."""
    vals = {}
    if not os.path.exists(path):
        return vals
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, _, v = line.partition("=")
            k, v = k.strip(), v.strip().strip('"').strip("'")
            vals[k] = v
            if into_environ:
                os.environ[k] = v
    return vals


def teleport_env():
    """Export short-lived Teleport provider credentials into this process.

    `tctl terraform env` prints `export TF_TELEPORT_...=...` and provisions a temporary
    bot in the cluster with a ~1 hour TTL. Terraform's Teleport provider reads these from
    the environment, so they must land in os.environ rather than a throwaway subshell.
    """
    txt = out("tctl terraform env")
    n = 0
    for key, val in re.findall(r"export\s+(\w+)=(.*)", txt):
        os.environ[key] = val.strip().strip('"').strip("'")
        n += 1
    if n == 0:
        raise RuntimeError("tctl terraform env produced no credentials -- is tsh logged in?")
    print(f"✅ Teleport provider credentials set ({n} vars), "
          f"addr={os.environ.get('TF_TELEPORT_ADDR')}")


def tf(args, *, cwd, **kwargs):
    return run(f"terraform {args}", cwd=cwd, **kwargs)


def write_if_changed(path, content):
    """Write only when content differs, so re-running a cell stays a no-op."""
    rel = os.path.relpath(path, REPO_DIR)
    if os.path.exists(path) and open(path).read() == content:
        print(f"unchanged: {rel}")
        return False
    with open(path, "w") as f:
        f.write(content)
    print(f"wrote: {rel}")
    return True


print("Helpers loaded. Repo:", REPO_DIR)

## Step 0 — Verify prerequisites

Everything below must already be installed. **No cell installs tooling for you** — if something
is missing the cell fails with the command to fix it.

| Tool | Why it's needed |
|---|---|
| `go` (1.24+) | Cross-compiles the extension and handler for `linux/arm64` |
| `make` | Drives the build: downloads `tbot`, produces both zips |
| `zip` | Packages the Lambda layer and function |
| `terraform` (1.5+) | Provisions the AWS and Teleport resources |
| `aws` | Invokes the function, reads CloudWatch logs |
| `tsh` / `tctl` | Teleport session, SPIFFE CA export, provider credentials |
| `ssh` | Configures Postgres on the EC2 instance |
| `openssl` | Inspects the certificates |
| `jq` | Optional, for ad-hoc JSON poking |

On macOS, `make` comes with the Xcode command line tools (`xcode-select --install`); the rest
are available via Homebrew.

The notebook's own Python needs are just `ipykernel` and `nbconvert` — see `requirements.txt`.
Use a virtualenv and point your IDE's project interpreter at it:

```
python3 -m venv .venv
.venv/bin/pip install -r requirements.txt
```

You also need an **active Teleport session** with rights to manage bots, roles, tokens, and
workload identities, plus AWS credentials able to create IAM roles, Lambda functions, EC2
instances, and security groups.

In [ ]:
REQUIRED = ["go", "make", "zip", "terraform", "aws", "tsh", "tctl", "ssh", "openssl"]
OPTIONAL = ["jq"]

missing = []
for tool in REQUIRED + OPTIONAL:
    path = shutil.which(tool)
    suffix = "" if tool in REQUIRED else "  (optional)"
    if path:
        print(f"{tool:11s} ✅ {path}{suffix}")
    else:
        print(f"{tool:11s} ❌ NOT FOUND{suffix}")
        if tool in REQUIRED:
            missing.append(tool)

if missing:
    raise RuntimeError(
        "Install these before continuing: " + ", ".join(missing)
        + "\n  macOS: brew install " + " ".join(m for m in missing if m != "make")
        + "\n  make ships with the Xcode command line tools: xcode-select --install"
    )

print()
run("go version")
run("terraform version | head -1")

# Python environment. PyCharm runs the notebook on the project interpreter, so a
# venv keeps these packages out of the system Python.
in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
print(f"\npython      {'✅' if in_venv else '⚠️ '} {sys.executable}")
print(f"            {'virtualenv' if in_venv else 'NOT a virtualenv -- see requirements.txt'}")

for mod, why in (("ipykernel", ""), ("nbconvert", " (Step 13 export will fail)")):
    try:
        __import__(mod)
        print(f"{mod:11s} ✅ importable")
    except ImportError:
        print(f"{mod:11s} ❌ missing -- pip install -r requirements.txt{why}")

## Step 1 — Configuration

Settings live in `.env` at the repo root. Create it before running this cell:

```
cp .env.example .env     # then set TELEPORT_PROXY_ADDR
```

| Key | Meaning |
|---|---|
| `TELEPORT_PROXY_ADDR` | **Required.** e.g. `example.teleport.sh:443` |
| `TELEPORT_AUTH_SERVER` | Address `tctl` connects to; usually the same. Blank reuses the proxy address |
| `AWS_REGION` | Defaults to `us-east-1` |
| `DEMO_NAME` | Base name for every created resource. Default `tbot-svid-demo` |
| `ARCH` | `arm64` (default) or `amd64` |
| `VPC_ID`, `SUBNET_ID` | Leave blank to auto-discover a VPC with a public subnet |
| `ADMIN_CIDR` | Leave blank to auto-detect your public IP as a `/32` |
| `AWS_ENV_FILE` | Optional path to a file of `AWS_*` exports, if you don't use the default credential chain |

`TELEPORT_VERSION` is deliberately **not** set here — Step 6 reads it from your live cluster, so
the `tbot` you ship always matches the cluster's major version.

### Pinning the cluster

If you are logged into **several Teleport proxies**, `tsh` and `tctl` follow
`~/.tsh/current-profile`, which may not be the cluster you want. This cell exports both pins
explicitly, from `.env`, for the rest of the notebook:

| Variable | Read by | Source |
|---|---|---|
| `TELEPORT_PROXY` | `tsh` | `TELEPORT_PROXY_ADDR` |
| `TELEPORT_AUTH_SERVER` | `tctl` | `TELEPORT_AUTH_SERVER`, defaulting to the proxy address |

**`tctl` ignores `TELEPORT_PROXY`** — that's why there are two. Setting only one pins half the
calls, which is worse than not pinning, because it looks handled.

They go into `os.environ`, so child processes inherit them; that's what pins the
`tctl auth export --type tls-spiffe` inside `database/setup.sh` too.

This matters because the failure is otherwise silent: exporting the wrong cluster's SPIFFE CA
into Postgres gives `certificate verify failed` on every connection while every piece of
configuration still looks right. The cell **raises** if `tctl` can't reach the cluster at all,
and warns if the cluster name differs from the proxy hostname — which is normal for self-hosted
clusters but worth a second look.

In [ ]:
env_path = os.path.join(REPO_DIR, ".env")
if not os.path.exists(env_path):
    raise FileNotFoundError(
        f"{env_path} not found. Create it from the template:\n"
        "    cp .env.example .env\n"
        "    then set TELEPORT_PROXY_ADDR and re-run this cell."
    )

cfg = load_dotenv(env_path)

# AWS credentials. Preference order: the default chain (~/.aws/credentials, SSO,
# instance profile), then AWS_ENV_FILE pointing outside the repo, then keys set
# inline in .env -- honoured, but warned about, since .env sits in the repo.
if cfg.get("AWS_ENV_FILE"):
    aws_env = os.path.expanduser(cfg["AWS_ENV_FILE"])
    if os.path.exists(aws_env):
        load_dotenv(aws_env, into_environ=True)
        print(f"loaded AWS credentials from {aws_env}")
    else:
        print(f"⚠️  AWS_ENV_FILE={aws_env} not found; using the default credential chain")

inline_keys = [k for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY",
                           "AWS_SESSION_TOKEN") if cfg.get(k)]
if inline_keys:
    for k in inline_keys:
        os.environ[k] = cfg[k]
    print(f"⚠️  using {', '.join(inline_keys)} set directly in .env")
    print("    .env lives inside the repo. Prefer the default credential chain, or")
    print("    AWS_ENV_FILE pointing outside it. Never commit real keys.")

TELEPORT_PROXY_ADDR = cfg.get("TELEPORT_PROXY_ADDR", "")
if not TELEPORT_PROXY_ADDR or TELEPORT_PROXY_ADDR.startswith("example."):
    raise ValueError(
        f"TELEPORT_PROXY_ADDR is not set in {env_path}\n"
        "    Edit that file, e.g. TELEPORT_PROXY_ADDR=mycluster.teleport.sh:443\n"
        "    then re-run this cell."
    )

AWS_REGION  = cfg.get("AWS_REGION") or "us-east-1"
DEMO_NAME   = cfg.get("DEMO_NAME") or "tbot-svid-demo"
ARCH        = cfg.get("ARCH") or "arm64"
LAMBDA_ARCH = "arm64" if ARCH == "arm64" else "x86_64"
KEY_PATH    = os.path.join(DB_DIR, f"{DEMO_NAME}.pem")

os.environ["AWS_REGION"] = AWS_REGION
os.environ.setdefault("AWS_DEFAULT_REGION", AWS_REGION)

# Pin every tsh and tctl call to this cluster. Without it they follow
# ~/.tsh/current-profile, which is the wrong cluster whenever you're logged into
# more than one -- and it fails silently in the worst way: Step 4 would export a
# different cluster's SPIFFE CA into Postgres, and every SVID would then be
# rejected while all the configuration still looked correct.
#
# tsh reads TELEPORT_PROXY; tctl ignores that and reads TELEPORT_AUTH_SERVER.
# Both are set -- one alone pins only half the calls. They live in os.environ so
# child processes inherit them, which is what pins the `tctl auth export` inside
# database/setup.sh as well.
TELEPORT_AUTH_SERVER = cfg.get("TELEPORT_AUTH_SERVER") or TELEPORT_PROXY_ADDR
proxy_host = TELEPORT_PROXY_ADDR.split(":")[0]

os.environ["TELEPORT_PROXY"] = TELEPORT_PROXY_ADDR
os.environ["TELEPORT_AUTH_SERVER"] = TELEPORT_AUTH_SERVER

reached = out("tctl status | awk '/^Cluster/ {print $2}'", check=False)
if not reached:
    raise RuntimeError(
        f"tctl could not reach {TELEPORT_AUTH_SERVER}.\n"
        f"    Run: tsh login --proxy={TELEPORT_PROXY_ADDR}"
    )

print(f"✅ tsh  -> TELEPORT_PROXY={TELEPORT_PROXY_ADDR}")
print(f"✅ tctl -> TELEPORT_AUTH_SERVER={TELEPORT_AUTH_SERVER}")
print(f"   cluster: {reached}")
# For Teleport Cloud the cluster name is the proxy host. Self-hosted clusters can
# legitimately differ, so this is a prompt to look rather than a failure.
if reached != proxy_host:
    print(f"⚠️  cluster name {reached!r} differs from proxy host {proxy_host!r} --"
          " normal for self-hosted, but confirm it's the cluster you intended")

print(f"\nregion={AWS_REGION}  name={DEMO_NAME}  arch={ARCH} (lambda: {LAMBDA_ARCH})")
print(f"ssh key path: {KEY_PATH}")

### Confirm both identities

This comes *after* configuration on purpose: `AWS_ENV_FILE` may supply the credentials, so
there is nothing meaningful to check until `.env` has been read.

Look at both outputs before provisioning anything — the account number and the Teleport cluster
are what everything below is created in.

In [ ]:
run("aws sts get-caller-identity")
run("tsh status | head -6")
run("tctl status | head -3")

## Step 2 — Discover AWS networking

The Postgres host needs a **public subnet**, and the Lambda deliberately runs with **no VPC
configuration**. That trips people up, so it's worth stating plainly:

A Lambda ENI never receives a public IP. A subnet whose default route is an internet gateway
therefore gives a Lambda **no internet access at all** — only a NAT gateway would, at roughly
\$32/month. Since tbot has to reach your Teleport proxy on 443, the function stays outside the
VPC and reaches Postgres over its public IP.

This cell finds a VPC with a public subnet and detects your IP for the SSH rule; both are
overridable in `.env`. It writes `terraform/db/terraform.tfvars`.

In [ ]:
VPC_ID     = cfg.get("VPC_ID", "")
SUBNET_ID  = cfg.get("SUBNET_ID", "")
ADMIN_CIDR = cfg.get("ADMIN_CIDR", "")

if not ADMIN_CIDR:
    ADMIN_CIDR = out("curl -s --max-time 10 https://checkip.amazonaws.com") + "/32"
    print(f"detected admin IP: {ADMIN_CIDR}")

if not SUBNET_ID:
    subnets = _json.loads(out("aws ec2 describe-subnets --output json"))["Subnets"]
    rtables = _json.loads(out("aws ec2 describe-route-tables --output json"))["RouteTables"]

    # A subnet is "public" when its effective route table sends 0.0.0.0/0 to an igw-.
    main_rt, explicit_rt = {}, {}
    for rt in rtables:
        via_igw = any(
            (r.get("GatewayId") or "").startswith("igw-")
            for r in rt.get("Routes", [])
            if r.get("DestinationCidrBlock") == "0.0.0.0/0"
        )
        for assoc in rt.get("Associations", []):
            if assoc.get("Main"):
                main_rt[rt["VpcId"]] = via_igw
            elif assoc.get("SubnetId"):
                explicit_rt[assoc["SubnetId"]] = via_igw

    for s in subnets:
        if VPC_ID and s["VpcId"] != VPC_ID:
            continue
        if explicit_rt.get(s["SubnetId"], main_rt.get(s["VpcId"], False)):
            VPC_ID, SUBNET_ID = s["VpcId"], s["SubnetId"]
            print(f"discovered public subnet {SUBNET_ID} "
                  f"({s['CidrBlock']}, {s['AvailabilityZone']}) in {VPC_ID}")
            break

if not SUBNET_ID:
    raise RuntimeError(
        "No public subnet found. Set VPC_ID and SUBNET_ID in .env, or create a subnet whose "
        "route table sends 0.0.0.0/0 to an internet gateway."
    )

db_tfvars = f"""# Generated by demo-walkthrough.ipynb -- edit .env instead.
aws_region       = "{AWS_REGION}"
vpc_id           = "{VPC_ID}"
subnet_id        = "{SUBNET_ID}"
admin_cidr       = "{ADMIN_CIDR}"
private_key_path = "{KEY_PATH}"
"""
write_if_changed(os.path.join(DB_DIR, "terraform.tfvars"), db_tfvars)

print(f"\nvpc={VPC_ID}  subnet={SUBNET_ID}  admin_cidr={ADMIN_CIDR}")

## Step 3 — Provision the Postgres host

`terraform/db` is a **separate root module** from the main demo, deliberately: the main module
needs `db_host` and the server CA, neither of which exists until this instance is running and
Step 4 has minted certificates. Separate state makes that ordering natural instead of requiring
`-target`.

Creates a `t4g.micro` (~\$6/month) with a public IP, a security group, and an ED25519 key pair.
`terraform apply` is inherently convergent, so re-running this cell is a no-op once the instance
exists — and `lifecycle { ignore_changes = [ami] }` stops a newly published AL2023 image from
replacing a running instance.

> The generated SSH private key is stored in `terraform/db/terraform.tfstate` **in plaintext**.
> State is gitignored; treat that directory as sensitive.

In [ ]:
tf("init -input=false", cwd=DB_DIR)
tf("apply -input=false -auto-approve", cwd=DB_DIR)

DB_HOST     = out("terraform output -raw public_ip", cwd=DB_DIR)
INSTANCE_ID = out("terraform output -raw instance_id", cwd=DB_DIR)
os.chmod(KEY_PATH, 0o600)

SSH_ARGV = ["ssh", "-i", KEY_PATH,
            "-o", "StrictHostKeyChecking=accept-new",
            "-o", "ConnectTimeout=10",
            f"ec2-user@{DB_HOST}"]

print(f"\n✅ instance {INSTANCE_ID} at {DB_HOST}")

In [ ]:
# Wait for sshd. A fresh instance needs roughly 20-40s before it accepts connections.
for attempt in range(30):
    probe = subprocess.run(SSH_ARGV + ["true"], capture_output=True,
                           text=True, env=os.environ)
    if probe.returncode == 0:
        print(f"✅ SSH ready after ~{attempt * 5}s")
        break
    print("  waiting for sshd...")
    time.sleep(5)
else:
    raise TimeoutError(
        f"SSH to {DB_HOST} never succeeded. Check that ADMIN_CIDR ({ADMIN_CIDR}) is still "
        "your public IP -- it changes if your network does."
    )

ssh("uname -m && cloud-init status")

## Step 4 — Certificates

`database/setup.sh` produces three things:

| File | Role |
|---|---|
| `spiffe-ca.pem` | Your cluster's SPIFFE CA, via `tctl auth export --type tls-spiffe`. Becomes Postgres's `ssl_ca_file` — **this is what makes Postgres trust Teleport-issued client certificates.** |
| `server-ca.pem` | A throwaway CA. Terraform embeds it in the Lambda env as `DB_SERVER_CA_PEM` so the function can verify the server. |
| `server.crt` / `server.key` | The server certificate, with `subjectAltName = IP:<db_host>`. |

Because `db_host` is an IP, the SAN is an `IP:` entry. That still supports
`DB_SSLMODE=verify-full`: the handler sets `tls.Config.ServerName` to `DB_HOST`, and Go checks
IP SANs when the name parses as an IP address. So server verification stays on.

**Idempotency:** regenerating certificates would force a Postgres restart and a Lambda
environment update, so this cell skips the work when the existing cert already covers the
current `db_host`.

In [ ]:
CERT_DIR   = os.path.join(REPO_DIR, "database", "certs")
server_crt = os.path.join(CERT_DIR, "server.crt")

if os.path.exists(server_crt):
    san = out(f"openssl x509 -in {server_crt} -noout -ext subjectAltName", check=False)
    if DB_HOST in san:
        print(f"✅ existing certificate already covers {DB_HOST} -- skipping regeneration")
    else:
        print(f"certificate does not cover {DB_HOST} -- regenerating")
        run(f"bash database/setup.sh {DB_HOST}")
else:
    run(f"bash database/setup.sh {DB_HOST}")

run(f"openssl x509 -in {server_crt} -noout -subject -ext subjectAltName")
run(f"openssl x509 -in {CERT_DIR}/spiffe-ca.pem -noout -subject -enddate")
run(f"openssl verify -CAfile {CERT_DIR}/server-ca.pem {server_crt}")

## Step 5 — Install Postgres on the instance

`database/deploy-postgres.sh` copies the certificates, `pg_hba.conf`, `init.sql`, and the
installer to the host, then runs `database/install-postgres.sh` there as root. No container
runtime is involved — it installs from the OS packages:

- installs `postgresql17-server` from the AL2023 repositories
- runs `initdb`, then places `server.crt` / `server.key` / `spiffe-ca.pem` with `postgres`
  ownership — Postgres refuses to start if its key file is group-readable
- appends `ssl = on`, the three certificate paths, `log_connections = on`, and
  **`listen_addresses = '*'`**
- installs the repo's `pg_hba.conf`, whose only remotely reachable line is
  `hostssl demo lambda_svid_demo … cert`
- creates database `demo`, then role `lambda_svid_demo` and table `lambda_invocations`

Two settings that are easy to miss: **`listen_addresses`** (a stock `initdb` binds localhost
only, which from the Lambda looks exactly like a security-group problem) and
**`createdb demo`** (`init.sql` creates only the role and table, not the database itself).

Every step is guarded, so re-running is safe.

In [ ]:
run(f"bash database/deploy-postgres.sh {DB_HOST} {KEY_PATH}")

In [ ]:
# Independent confirmation that 5432 is reachable from outside AWS.
with socket.socket() as s:
    s.settimeout(5)
    reachable = s.connect_ex((DB_HOST, 5432)) == 0

print(("✅" if reachable else "❌") + f" TCP {DB_HOST}:5432 reachable")
if not reachable:
    raise RuntimeError("5432 unreachable -- check the security group and listen_addresses")

## Step 6 — Build the layer and the function

`make` produces two artifacts:

| Artifact | Contents |
|---|---|
| `build/tbot-layer.zip` | `extensions/tbot-extension` + `bin/tbot` |
| `build/function.zip` | `bootstrap`, the handler |

Those internal paths are load-bearing: Lambda only runs extensions found under
`/opt/extensions/`, and the extension defaults to `/opt/bin/tbot`.

`TELEPORT_VERSION` is read from your **live cluster**, so the `tbot` you ship always matches its
major version. The cell also checks the layer against Lambda's **50 MB inline upload limit** —
beyond that the layer has to be uploaded via S3 instead of inline.

In [ ]:
TELEPORT_VERSION = out("tctl status | awk '/^Version/ {print $2}'")
print(f"cluster version {TELEPORT_VERSION}, building for {ARCH}\n")

run("go mod tidy")
run(f"make TELEPORT_VERSION={TELEPORT_VERSION} ARCH={ARCH}")

layer_zip = os.path.join(REPO_DIR, "build", "tbot-layer.zip")
fn_zip    = os.path.join(REPO_DIR, "build", "function.zip")
INLINE_LIMIT = 50 * 1024 * 1024

for path in (layer_zip, fn_zip):
    print(f"{os.path.basename(path):20s} {os.path.getsize(path) / 1e6:6.1f} MB")

if os.path.getsize(layer_zip) > INLINE_LIMIT:
    raise RuntimeError(
        f"tbot-layer.zip is {os.path.getsize(layer_zip) / 1e6:.1f} MB, over Lambda's 50 MB "
        "inline upload limit. Switch aws_lambda_layer_version to an S3-backed upload."
    )

print("\n✅ both zips are within Lambda's inline upload limit")
run("unzip -l build/tbot-layer.zip | tail -6")

## Step 7 — Deploy the Lambda and the Teleport resources

Nine resources: IAM role, log group, layer, and function on the AWS side; `workload_identity`,
`role`, `bot`, and `provision_token` on the Teleport side.

The two policy layers that make this zero-trust rather than merely zero-password:

- **`provision_token`** — `join_method: iam`, restricted to your account **and** the exact
  assumed-role ARN `…:assumed-role/<name>-lambda/<name>`. Only that Lambda can become the bot.
- **`workload_identity`** — issuance additionally requires `join.iam.arn` to equal that ARN
  **and** `workload.unix.binary_path` to be `/var/task/bootstrap`, which tbot attests over
  `SO_PEERCRED` plus `/proc`. Another process inside the same sandbox cannot obtain the SVID.

`teleport_env()` runs `tctl terraform env`, which provisions a **temporary bot** in your cluster
(~1 h TTL) and exports credentials for the provider. That is a write to your cluster beyond the
demo's own resources; those temporary bots expire on their own.

In [ ]:
main_tfvars = f"""# Generated by demo-walkthrough.ipynb -- edit .env instead.
aws_region          = "{AWS_REGION}"
teleport_proxy_addr = "{TELEPORT_PROXY_ADDR}"
name                = "{DEMO_NAME}"
lambda_architecture = "{LAMBDA_ARCH}"

db_host               = "{DB_HOST}"
db_sslmode            = "verify-full"
db_server_ca_pem_file = "../database/certs/server-ca.pem"

# The Lambda stays OUT of the VPC: a Lambda ENI never gets a public IP, so an
# internet-gateway route cannot give it egress to the Teleport proxy.
# subnet_ids         = []
# security_group_ids = []
"""
write_if_changed(os.path.join(TF_DIR, "terraform.tfvars"), main_tfvars)

teleport_env()
tf("init -input=false", cwd=TF_DIR)
tf("apply -input=false -auto-approve", cwd=TF_DIR)
run("terraform output", cwd=TF_DIR)

In [ ]:
# Confirm the Teleport resources exist in the cluster, not merely in Terraform state.
wi = _json.loads(out(f"tctl get workload_identity/{DEMO_NAME} --format=json"))[0]
spiffe = wi["spec"]["spiffe"]
print(f"workload_identity : {wi['metadata']['name']}")
print(f"  spiffe id       : {spiffe['id']}")
print(f"  common_name     : {spiffe['x509']['subject_template']['common_name']}")
print(f"  maximum_ttl     : {spiffe['x509']['maximum_ttl']}")
for cond in wi["spec"]["rules"]["allow"][0]["conditions"]:
    print(f"  require         : {cond['attribute']} == {cond['eq']['value']}")

tokens = _json.loads(out("tctl get tokens --format=json"))
tok = next(t for t in tokens if t["metadata"]["name"] == f"{DEMO_NAME}-iam")
print(f"\nprovision_token   : {tok['metadata']['name']}")
print(f"  join_method     : {tok['spec']['join_method']}")
print(f"  allow           : {tok['spec']['allow']}")

print("\nNo bot instance exists yet -- tbot only runs during a Lambda cold start.")
run("tctl bots instances ls | head -3")

## Step 8 — Run the demo

The first invoke is a **cold start**: the extension registers with the Extensions API, writes
`/tmp/tbot/tbot.yaml`, starts tbot, and blocks INIT until tbot's `/readyz` returns 200. Lambda
does not deliver the first invoke until every extension has called `/event/next`, so the handler
never races tbot.

Watch for `current_user = lambda_svid_demo` and a populated `ssl_client_dn` — that second field
is the proof that Postgres authenticated a *certificate*, not something else.

In [ ]:
def invoke():
    """Invoke the function and return its parsed JSON response."""
    raw = out(
        f"aws lambda invoke --region {AWS_REGION} --function-name {DEMO_NAME}"
        " --cli-binary-format raw-in-base64-out --payload '{}' /dev/stdout"
    )
    # The CLI appends its own {"StatusCode": ...} object after the payload, so decode
    # just the first JSON value rather than the whole stream.
    payload, _ = _json.JSONDecoder().raw_decode(raw)
    return payload


resp = invoke()
print(_json.dumps(resp, indent=2))

svid = resp["svid"]
db = resp.get("database") or {}
if resp.get("error"):
    raise RuntimeError(f"handler reported: {resp['error']}")
assert db.get("current_user") == "lambda_svid_demo", f"unexpected DB user: {db}"
assert db.get("ssl_client_dn"), "ssl_client_dn empty -- Postgres saw no client certificate"

parse = lambda s: datetime.datetime.fromisoformat(s.replace("Z", "+00:00"))
ttl = parse(svid["not_after"]) - parse(svid["not_before"])

display(Markdown(f"""
| | |
|---|---|
| SPIFFE ID | `{svid['spiffe_id']}` |
| Subject | `{svid['subject']}` |
| Issuer | `{svid['issuer'].split(',')[-1].strip()}` |
| Serial | `{svid['serial']}` |
| TTL | **{ttl}** |
| Reused from memory | {svid['reused']} (expires in {svid['expires_in']}) |
| `current_user` | **`{db['current_user']}`** |
| `ssl_client_dn` | `{db['ssl_client_dn']}` |
| Postgres version | {db.get('server_version')} |
"""))

## Step 9 — The evidence trail

Four independent confirmations. The Postgres log is the most persuasive for an audience that
won't take the function's self-report on faith.

On SVID reuse: by default the handler holds a `workloadapi.X509Source` in a package variable,
which survives freeze/thaw, so warm invokes are served from memory and `fetch_svid` drops to
roughly zero. tbot renews ahead of expiry and pushes the replacement over the Workload API
stream. The handler still checks remaining validity on every invoke, because a frozen sandbox
renews nothing and a pushed update may not have been processed yet when the invoke starts —
see the comment block in `cmd/handler/main.go`.

Set `svid_cache = false` in tfvars to fetch fresh on every invoke and compare.

In [ ]:
# 9a. The bot instance that did not exist before the first invoke.
run(f"tctl bots instances ls | grep -E 'ID|{DEMO_NAME}'")

In [ ]:
# 9b. The cold start: extension registering, then tbot joining via IAM.
run(f"aws logs tail /aws/lambda/{DEMO_NAME} --since 15m --format short"
    " | grep -iE 'INIT_START|tbot-extension|Attempting to join|Fetched new bot identity'"
    " | head -12")

In [ ]:
# 9c. A second invoke: is the SVID reused from memory, or freshly issued?
second = invoke()
a, b = resp["svid"], second["svid"]

for n, s, t in ((1, a, resp), (2, b, second)):
    print(f"invoke {n}: serial={s['serial']} reused={s['reused']:<5} "
          f"expires_in={s['expires_in']:<10} fetch_svid={t['timings_ms']['fetch_svid']}ms")

if b["reused"]:
    assert a["serial"] == b["serial"], "reused=true but the serial changed"
    print("\n✅ SVID reused from the in-memory X509Source -- no round trip to Teleport.")
    print("   The first invoke paid for the issuance; this one paid nothing.")
else:
    assert a["serial"] != b["serial"], "not reused, so a new serial was expected"
    print("\n✅ fresh SVID issued (SVID_CACHE=false, or the cached one was near expiry)")

print(f"   rows recorded: {second['database']['total_invocations_recorded']}")

In [ ]:
# 9d. Postgres's own account of it: cert auth over TLS, no password involved.
ssh('sudo bash -c "grep -hE \\"connection (received|authorized)\\" '
    '$(ls -t /var/lib/pgsql/data/log/*.log | head -1) | tail -8"')

## Step 10 — Negative tests

The demo is only interesting if the failure modes actually fail.

In [ ]:
# 10a. No client certificate -> rejected. TCP connects; authentication does not.
probe = ssh(
    f'psql "host={DB_HOST} dbname=demo user=lambda_svid_demo sslmode=require" -c "select 1"',
    check=False, show=False)
print((probe.stdout + probe.stderr).strip() or "(no output)")
print("\n✅ expected: authentication failed -- no client certificate was presented")

In [ ]:
# 10b. The function holds no secrets: only names, plus one public CA certificate.
env_vars = _json.loads(out(
    f"aws lambda get-function-configuration --function-name {DEMO_NAME}"
    f" --region {AWS_REGION} --query Environment.Variables"))

for key, val in sorted(env_vars.items()):
    shown = f"<{len(val)} bytes of PEM>" if "PEM" in key else val
    print(f"  {key:24s} = {shown}")

print("\n✅ no DB password, no Teleport token secret, no private key")
print("   DB_SERVER_CA_PEM is a public certificate, used to verify the server")

## Step 11 — What was built

| Layer | Resource |
|---|---|
| AWS | Lambda function (`provided.al2023`, arm64, no VPC configuration) |
| AWS | Lambda layer: the tbot extension plus the `tbot` binary |
| AWS | IAM execution role — **no Teleport or database permissions**; its *identity* is what Teleport trusts |
| AWS | EC2 `t4g.micro` running Postgres 17 with certificate-only authentication |
| Teleport | `workload_identity` — what the SVID looks like, and who may receive it |
| Teleport | `role` — lets the bot issue that one workload identity, scoped by label |
| Teleport | `bot` — the machine identity tbot runs as |
| Teleport | `provision_token` — IAM join, pinned to the Lambda's assumed-role ARN |

**Why there are no secrets.** `sts:GetCallerIdentity` requires no IAM permission, so the
execution role carries nothing but basic logging. tbot signs a `GetCallerIdentity` request using
the credentials Lambda injects, and Teleport verifies the resulting ARN against the token.
Identity derives from *where the code runs*, not from anything it stores.

**Policy on both sides.** Teleport decides *who gets* an identity (`join.iam.arn`,
`workload.unix.binary_path`). Postgres decides *what it can do* (`GRANT`s to
`lambda_svid_demo`). Neither side trusts the other's decision.

**Freeze/thaw.** tbot is frozen along with the sandbox between invokes. After a long idle its
internal certificates may have expired; with IAM join it simply re-joins on the next renewal.
The handler fetches a fresh SVID every invoke rather than caching one.

### Extensions worth trying

- Give **Postgres its own SVID** (tbot on the DB host plus a `workload-identity-x509` output).
  Both sides then verify each other with SPIFFE and the throwaway server CA disappears.
- Use `workloadapi.X509Source` with a connection pool to reuse connections across warm invokes,
  rotating as SVIDs renew.
- Use a **JWT-SVID** to call an HTTP API, or AWS via Roles Anywhere, from the same extension.
- Package it as a container-image Lambda to sidestep the 250 MB unzipped limit.

## Step 12 — Teardown

Destroys all 17 resources: 9 from the main module, 8 from the database module — including the
EC2 instance and its EBS volume, the key pair, and the local `.pem`.

**Gated deliberately.** With `DESTROY = False` the cell shows a destroy *plan* and changes
nothing. Set `DESTROY = True` to arm it.

Your pre-existing VPC, subnet, and internet gateway are never touched; the notebook only read
them. Also not covered: the temporary `tctl-terraform-env-*` bots (they expire on their own) and
local files under `database/certs/` and `build/`.

In [ ]:
DESTROY = False   # set True to tear everything down

teleport_env()
if DESTROY:
    tf("destroy -input=false -auto-approve", cwd=TF_DIR)
    tf("destroy -input=false -auto-approve", cwd=DB_DIR)
    print("\n✅ all resources destroyed")
else:
    print("Teardown disarmed. Set DESTROY = True and re-run this cell.\n")
    tf("plan -destroy -input=false", cwd=TF_DIR)
    tf("plan -destroy -input=false", cwd=DB_DIR)

## Step 13 — Export the executed notebook

Renders this notebook, outputs included, to a timestamped HTML file — a convenient artifact to
share after a demo run.

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
html_file = f"demo-walkthrough-executed-{timestamp}.html"
run(f"{sys.executable} -m jupyter nbconvert --to html demo-walkthrough.ipynb "
    f"--output {html_file}")
print(f"✅ {os.path.join(REPO_DIR, html_file)}")